In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

In [2]:
# BƯỚC 1:  ĐỌC FILE VÀ TIỀN XỬU LÝ DATA ---
df_raw = pd.read_csv('Metro_Interstate_Traffic_Volume.csv')
df_raw['date_time'] = pd.to_datetime(df_raw['date_time']) # gán trục thời gian

# Lọc trùng thời gian theo weather 
# Ưu tiên giữ lại các dòng có thời tiết xấu vì nó ảnh hưởng giao thông rõ rệt hơn
weather_priority = ['Thunderstorm', 'Snow', 'Rain', 'Drizzle', 'Mist', 'Haze', 'Fog']
df_raw['priority'] = df_raw['weather_main'].apply(lambda x: weather_priority.index(x) if x in weather_priority else 99)
df_raw = df_raw.sort_values(by=['date_time', 'priority']) # sắp xếp lại thời gian

# Xóa trùng, chỉ giữ lại 1 dòng duy nhất cho mỗi giờ
df_raw = df_raw.drop_duplicates(subset=['date_time'], keep='first')
df_raw = df_raw.drop(columns=['priority']) # Xóa cột phụ sau khi dùng xong

# Tính khoảng cách giữa 2 mốc thời gian liên tiếp
df_raw['time_diff'] = df_raw['date_time'].diff()

# Quy định gap lớn: lớn hơn 24 giờ thì xem là đứt đoạn lớn
gap_threshold = pd.Timedelta(hours=24)

# Đánh dấu đoạn mới
df_raw['new_segment'] = (df_raw['time_diff'] > gap_threshold).astype(int)

df_raw['segment_id'] = df_raw['new_segment'].cumsum()

# Tóm tắt các segment
segment_summary = df_raw.groupby('segment_id').agg(
    start_time=('date_time', 'min'),
    end_time=('date_time', 'max'),
    n_rows=('date_time', 'count')
).reset_index()

# Tính độ dài theo ngày
segment_summary['duration_days'] = (
    segment_summary['end_time'] - segment_summary['start_time']
).dt.days

display(segment_summary)

,segment_id,start_time,end_time,n_rows,duration_days
0,0,2012-10-02 09:00:00,2013-01-29 00:00:00,2664,118
1,1,2013-01-30 01:00:00,2013-08-31 00:00:00,4675,212
2,2,2013-09-01 23:00:00,2013-09-03 02:00:00,26,1
3,3,2013-09-04 08:00:00,2013-10-27 01:00:00,766,52
4,4,2013-11-06 04:00:00,2014-04-29 08:00:00,4057,174
5,5,2014-05-04 05:00:00,2014-07-27 16:00:00,1549,84
6,6,2014-07-30 09:00:00,2014-08-02 02:00:00,63,2
7,7,2014-08-03 11:00:00,2014-08-08 01:00:00,98,4
8,8,2015-06-11 20:00:00,2015-06-14 20:00:00,7,3
9,9,2015-06-19 18:00:00,2015-06-20 18:00:00,2,1


## CHỌN SEGMENT CHO BÀI 

In [3]:
# ==========================================
# CHỌN SEGMENT CHÍNH THỨC
# ==========================================

selected_segment_id = 12 # seg 12

df_selected = (
    df_raw[df_raw['segment_id'] == selected_segment_id]
    .copy()
)

# Xóa cột phụ dùng để detect gap
df_selected = df_selected.drop(
    columns=['time_diff', 'new_segment'],
    errors='ignore'
)

# Reset index
df_selected = df_selected.reset_index(drop=True)

print("Segment được chọn:", selected_segment_id)
print("Số dòng:", len(df_selected))

print("Thời gian bắt đầu:")
print(df_selected['date_time'].min())

print("Thời gian kết thúc:")
print(df_selected['date_time'].max())



df_selected.to_csv(
    "traffic_raw_selected_segment.csv",
    index=False
)

print("Đã lưu traffic_raw_selected_segment.csv")

Segment được chọn: 12
Số dòng: 23979
Thời gian bắt đầu:
2015-10-27 08:00:00
Thời gian kết thúc:
2018-09-30 23:00:00
Đã lưu traffic_raw_selected_segment.csv


In [4]:
# Tiếp tục tiền xử lý cơ bản cho segment mới chọn 

# --- SỬ DỤNG TỆP DATA "traffic_raw_selected_segment.csv"  LÀM DATA CHO TRAIN + MÔ PHỎNG

df = pd.read_csv('traffic_raw_selected_segment.csv')
df['date_time'] = pd.to_datetime(df['date_time'])
df= df.sort_values(by=['date_time'])

# Tạo danh sách các ngày thực sự là ngày lễ (Lấy TRƯỚC khi resample)
temp_dates = df['date_time'].dt.date
holiday_dates = temp_dates[df['holiday'].notnull() & (df['holiday'].astype(str) != 'None')].unique()

# Ép về chuỗi liên tục và nội suy số
df = df.set_index('date_time').resample('h').asfreq()
cols_num = ['traffic_volume', 'temp', 'rain_1h', 'snow_1h']
df[cols_num] = df[cols_num].interpolate(method='linear')
# Làm tròn và ép kiểu về Integer
df['traffic_volume'] = df['traffic_volume'].round().astype(int)
df = df.reset_index()


# --- TÍNH TOÁN FEATURES thời gian cơ bản (XỬ LÝ LỄ ĐÚNG CÁCH) ---

# --- TÍNH TOÁN CÁC BIẾN TĨNH ---
# 1. Gán is_holiday (So khớp với danh sách holiday_dates đã lấy ở Bước 1)
df['is_holiday'] = df['date_time'].dt.date.isin(holiday_dates).astype(int)
# Xóa holiday text cũ
df.drop(columns=['holiday'], inplace=True, errors='ignore')


# 2. Tính Peak và Hour display
hour_int = df['date_time'].dt.hour
df['hour'] = hour_int.apply(lambda x: f"{x}:00")
df['is_peak'] = hour_int.isin([7, 8, 9, 16, 17, 18]).astype(int)
df['day_of_week'] = df['date_time'].dt.dayofweek

# 3. Điền khuyết Weather và dọn dẹp
cat_cols = ['weather_main', 'weather_description']

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].ffill().bfill()
# Kiểm tra nhanh: Nếu số 0 nhiều hơn số 1 là được
print("Thống kê cột is_holiday:")
print(df['is_holiday'].value_counts())

Thống kê cột is_holiday:
is_holiday
0    24928
1      744
Name: count, dtype: int64


In [5]:
# Chia tệp train - test : 80-20
split_point = int(len(df) * 0.8)

train_df = df.iloc[:split_point].copy()
test_df  = df.iloc[split_point:].copy()

# lưu file nếu cần
train_df.to_csv('traffic_train.csv', index=False)
test_df.to_csv('traffic_test.csv', index=False)

print("Chia dữ liệu xong:")
print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Chia dữ liệu xong:
Train size: 20537
Test size: 5135


In [6]:
# --- BƯỚC 3: TẠO CÁC FEATURE ĐỘNG CHO TỆP TRAFFIC_TRAIN PHỤC VỤ huấn uyện và lựa chọn MÔ HÌNH ---

df_train=pd.read_csv('traffic_train.csv')
# Đảm bảo dữ liệu được sắp xếp theo thời gian
df_train = df_train.sort_values('date_time').reset_index(drop=True)

# 1. Lag feature: lưu lượng ở thời điểm trước đó
df_train['lag_1'] = df_train['traffic_volume'].shift(1)

# 2. Biến thiên lưu lượng so với thời điểm trước
df_train['volume_change'] = df_train['traffic_volume'] - df_train['lag_1']

# 3. Trung bình trượt 3 giờ
df_train['avg_3h'] = df_train['traffic_volume'].rolling(window=3).mean()

# 4. Trung bình trượt 24 giờ
df_train['avg_24h'] = df_train['traffic_volume'].rolling(window=24).mean()

# 5. Độ lệch chuẩn trượt 3 giờ
df_train['rolling_std_3h'] = df_train['traffic_volume'].rolling(window=3).std()

# 6. Tỷ lệ giữa trung bình ngắn hạn và trung bình dài hạn
# Thêm 1e-6 để tránh chia cho 0
df_train['volume_deviation'] = df_train['traffic_volume'] - df_train['avg_24h'] 

train_features = df_train.dropna(subset=[
    'lag_1',
    'volume_change',
    'avg_3h',
    'avg_24h',
    'rolling_std_3h',
    'volume_deviation'
]).reset_index(drop=True)

# Làm tròn nhẹ cho dễ đọc
feature_cols_new = [
    'lag_1',
    'volume_change',
    'avg_3h',
    'avg_24h',
    'rolling_std_3h',
    'volume_deviation'
]

df_train[feature_cols_new] = df_train[feature_cols_new].round(4)


# Kiểm tra số dòng bị mất
print("Số dòng train ban đầu:", len(df_train))
print("Số dòng train sau khi tạo feature:", len(train_features))
print("Số dòng bị loại:", len(df_train) - len(train_features))

print("\nKiểm tra missing:")
print(train_features[feature_cols_new].isnull().sum())


print("\nMissing values sau khi tạo feature:")
print(df_train[feature_cols_new].isnull().sum())


# LƯU FILE 
train_features.to_csv("traffic_train_with_features.csv",index=False )
print("\nĐã lưu traffic_train_with_features.csv thành công!")

Số dòng train ban đầu: 20537
Số dòng train sau khi tạo feature: 20514
Số dòng bị loại: 23

Kiểm tra missing:
lag_1               0
volume_change       0
avg_3h              0
avg_24h             0
rolling_std_3h      0
volume_deviation    0
dtype: int64

Missing values sau khi tạo feature:
lag_1                1
volume_change        1
avg_3h               2
avg_24h             23
rolling_std_3h       2
volume_deviation    23
dtype: int64

Đã lưu traffic_train_with_features.csv thành công!
